In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install datasets transformers torch feedparser newspaper3k readability-lxml beautifulsoup4 requests tqdm


In [ ]:
from datasets import load_dataset
import gc

# Load only XSum (smaller and better for summarization)
# Use only 10% of training data to fit in Colab memory
xsum_train = load_dataset("xsum", split="train[:10%]")
xsum_val = load_dataset("xsum", split="validation[:5%]")
print("XSum train samples:", len(xsum_train))
print("XSum validation samples:", len(xsum_val))


In [ ]:
from transformers import BartTokenizer
tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")


In [ ]:
def preprocess_batch(examples):
    inputs = examples["document"]
    targets = examples["summary"]
    
    model_inputs = tokenizer(
        inputs,
        max_length=512,  # Reduced from 1024 to save memory
        truncation=True,
        padding="max_length"
    )
    labels = tokenizer(
        targets,
        max_length=64,  # Reduced from 128
        truncation=True,
        padding="max_length"
    )
    
    return {
        "input_ids": model_inputs["input_ids"],
        "attention_mask": model_inputs["attention_mask"],
        "labels": labels["input_ids"]
    }

# Tokenize datasets
xsum_tok = xsum_train.map(
    preprocess_batch,
    batched=True,
    remove_columns=xsum_train.column_names
)
xsum_val_tok = xsum_val.map(
    preprocess_batch,
    batched=True,
    remove_columns=xsum_val.column_names
)

print("Tokenized train samples:", len(xsum_tok))
print("Tokenized val samples:", len(xsum_val_tok))

# Clear original data
del xsum_train
del xsum_val
gc.collect()


In [ ]:
# Use single dataset
combined_train = xsum_tok
combined_validation = xsum_val_tok
print("Train size:", len(combined_train))
print("Val size:", len(combined_validation))


In [ ]:
from torch.utils.data import DataLoader
import torch

def collate_fn(batch):
    return {
        "input_ids": torch.stack([torch.tensor(item["input_ids"]) for item in batch]),
        "attention_mask": torch.stack([torch.tensor(item["attention_mask"]) for item in batch]),
        "labels": torch.stack([torch.tensor(item["labels"]) for item in batch])
    }

train_loader = DataLoader(
    combined_train,
    batch_size=1,  # Small batch size for Colab
    shuffle=True,
    collate_fn=collate_fn
)
val_loader = DataLoader(
    combined_validation,
    batch_size=1,
    collate_fn=collate_fn
)
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")


In [ ]:
from transformers import BartForConditionalGeneration
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = BartForConditionalGeneration.from_pretrained("facebook/bart-large")
model = model.to(device)
print("Model loaded!")


In [ ]:
epochs = 1
learning_rate = 3e-5
warmup_steps = 100
gradient_accumulation_steps = 8  # Effective batch size = 1 * 8 = 8


In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optimizer = AdamW(model.parameters(), lr=learning_rate)
total_steps = len(train_loader) * epochs // gradient_accumulation_steps
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
print(f"Total training steps: {total_steps}")


In [ ]:
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler

# Enable gradient checkpointing to save memory
model.gradient_checkpointing_enable()

# Mixed precision training
scaler = GradScaler()

model.train()

for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    epoch_loss = 0
    
    for step, batch in enumerate(tqdm(train_loader)):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Mixed precision forward pass
        with autocast():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss
        
        # Scale loss and backward
        scaled_loss = loss / gradient_accumulation_steps
        scaler.scale(scaled_loss).backward()
        
        if (step + 1) % gradient_accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
        
        epoch_loss += loss.item()
        
        # Clear CUDA cache periodically
        if step % 100 == 0:
            torch.cuda.empty_cache()
    
    print(f"Epoch Loss: {epoch_loss:.4f}")


In [ ]:
model.save_pretrained("/content/drive/MyDrive/fine_tuned_bart_news")
tokenizer.save_pretrained("/content/drive/MyDrive/fine_tuned_bart_news")
print("Model saved!")
